In [1]:
import pandas as pd
import psycopg2
from datetime import datetime, timedelta
import warnings
import traceback

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

warnings.filterwarnings('ignore', category=UserWarning, message="pandas only supports SQLAlchemy connectable")

# MATCH_ID      = 11000000000
# EVENT_ID      = 12000000000
# DECK_ID       = 13000000000
# EVENT_TYPE_ID = 14000000000
# LOAD_RPT_ID   = 15000000000
# REJ_ID        = 16000000000

In [2]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [3]:
def get_df(query, vars=()):
    conn = psycopg2.connect(
        host=credentials[0],
        port=credentials[1],
        user=credentials[2],
        password=credentials[3],
        database=credentials[4]
    )

    df = pd.read_sql(query,conn,params=vars)

    conn.close()
    return df

In [8]:
def get_max_id():
    conn = psycopg2.connect(
        host=credentials[0],
        port=credentials[1],
        user=credentials[2],
        password=credentials[3],
        database=credentials[4]
    )

    cursor = conn.cursor()

    query1 = """
        SELECT max("EVENT_ID")
        FROM "EVENTS"
    """
    query2 = """
        SELECT max("MATCH_ID")
        FROM "MATCHES"
    """

    cursor.execute(query1)
    max_event_id = cursor.fetchone()[0]

    cursor.execute(query2)
    max_match_id = cursor.fetchone()[0]

    conn.close()
    return max_event_id, max_match_id

In [4]:
def parse_matchup_sheet(sheet,gid,match_id_start=11000000000,event_id_start=12000000000,start_date=None,end_date=None):
    def abstract_events(df_events,format):
        query = """
            SELECT *
            FROM "VALID_EVENT_TYPES"
            WHERE "FORMAT" = %s
        """

        df = pd.merge(left=df_events,right=get_df(query,(format,)),left_on=['EVENT_TYPE'],right_on=['EVENT_TYPE'])

        return df[['EVENT_ID','EVENT_DATE','EVENT_TYPE_ID']]

    def abstract_decks(df_matches,format):
        query = """
            SELECT *
            FROM "VALID_DECKS"
            WHERE "FORMAT" = %s
        """

        df_decks = get_df(query,(format,))

        df = pd.merge(left=df_matches,right=df_decks,left_on=['P1_ARCH','P1_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'])
        df.rename(columns={'DECK_ID':'P1_DECK_ID'},inplace=True)

        df = pd.merge(left=df,right=df_decks,left_on=['P2_ARCH','P2_SUBARCH'],right_on=['ARCHETYPE','SUBARCHETYPE'])
        df.rename(columns={'DECK_ID':'P2_DECK_ID'},inplace=True)

        return df[['MATCH_ID','P1','P2','P1_WINS','P2_WINS','MATCH_WINNER','P1_DECK_ID','P2_DECK_ID','P1_NOTE','P2_NOTE','EVENT_ID']]
    
    sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(sheet_url)

    # Rename columns.
    df.columns = ['P1','P2','P1_WINS','P2_WINS','WINNER1','WINNER2','P1_ARCH','P2_ARCH','P1_SUBARCH','P2_SUBARCH','P1_NOTE','P2_NOTE','EVENT_DATE','EVENT_TYPE']

    # Replace null values with 'NA' string.
    df.fillna({'P1_ARCH':'NA','P2_ARCH':'NA','P1_SUBARCH':'NA','P2_SUBARCH':'NA','P1_NOTE':'NA','P2_NOTE':'NA'},inplace=True)

    # Format EVENT_DATE column.
    df['EVENT_DATE'] = pd.to_datetime(df['EVENT_DATE'],yearfirst=False,format='mixed')

    # Handle empty EVENT_DATE values by forward-filling.
    df['EVENT_DATE'] = df['EVENT_DATE'].ffill()

    # Add 7-14 day lag time in case data is updated/corrected soon after upload.
    if start_date is None:
        # start_date = datetime.today().date() - timedelta(days=14)
        start_date = datetime(2024, 8, 24).date()
        
    if end_date is None:
        # end_date = datetime.today().date() - timedelta(days=7)
        end_date = datetime.today().date() + timedelta(days=1)
        
    df = df[(df['EVENT_DATE'] >= pd.to_datetime(start_date)) & (df['EVENT_DATE'] < pd.to_datetime(end_date))]

    # Adding Event_IDs.
    count = event_id_start
    df['EVENT_ID'] = 0
    for index, row in reversed(list(df.iterrows())):
        df.at[index,'EVENT_ID'] = count
        if pd.notna(row['EVENT_TYPE']):
            count += 1

    # Format EVENT_TYPE values.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].str.upper().str.strip()

    # Handle empty EVENT_TYPE values by forward-filling.
    df['EVENT_TYPE'] = df['EVENT_TYPE'].ffill()

    # Ignore events with incomplete data.
    events_to_ignore = df[df['P1_WINS'].isnull()].groupby(['EVENT_ID']).agg({'P1':'count'}).reset_index()['EVENT_ID'].tolist()
    # events_to_ignore = [1000007]
    df = df[~df.EVENT_ID.isin(events_to_ignore)]

    # Strip whitespace from player/deck names.
    df.P1 = df.P1.str.strip()
    df.P2 = df.P2.str.strip()
    df.P1_ARCH = df.P1_ARCH.str.strip().str.upper()
    df.P2_ARCH = df.P2_ARCH.str.strip().str.upper()
    df.P1_SUBARCH = df.P1_SUBARCH.str.strip().str.upper()
    df.P2_SUBARCH = df.P2_SUBARCH.str.strip().str.upper()
    df.P1_NOTE = df.P1_NOTE.str.strip().str.upper()
    df.P2_NOTE = df.P2_NOTE.str.strip().str.upper()

    # Format No Show deck name values.
    for index, row in df.iterrows():
        if row['P1_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P1_SUBARCH'] = 'NO SHOW'
        if row['P2_SUBARCH'].strip().upper() == 'NO SHOW':
            df.at[index,'P2_SUBARCH'] = 'NO SHOW'

    # Replace Winner1/2 columns with single Match_Winner column.
    df['MATCH_WINNER'] = df.apply(lambda row: 'P1' if ((row['WINNER1'] == 1) & (row['WINNER2'] == 0)) else ('P2' if ((row['WINNER1'] == 0) & (row['WINNER2'] == 1)) else 'NA'), axis=1)
    df.drop(columns=['WINNER1','WINNER2'],inplace=True)

    # Make these kind of corrections post-ETL.
    # EVENT_ID 1000067 should be OTHER.
    # df.loc[df['EVENT_ID'] == 1000067,'EVENT_TYPE'] = 'OTHER'.

    # Convert P1/P2_WINS from float to int.
    df['P1_WINS'] = df['P1_WINS'].astype(int)
    df['P2_WINS'] = df['P2_WINS'].astype(int)

    # Abstract out Event info into its own table.
    df_events = df.groupby(['EVENT_ID','EVENT_DATE']).agg({'EVENT_TYPE':'last'}).reset_index()

    # Calculate MATCH_IDs for each pair of rows that apply to the same match.
    df['match_key'] = df.apply(lambda row: frozenset([row['P1'],row['P2'],row['EVENT_ID']]), axis=1)
    df = df.reset_index()
    df = df.sort_values(by=['match_key','index'])
    df["group_id"] = df.groupby(['match_key']).cumcount() // 2
    df["MATCH_ID"] = (df.groupby(['match_key','group_id']).ngroup() + match_id_start)
    df = df.sort_values(by=['index'])
    df = df.drop(columns=['match_key','group_id','index'])

    return abstract_decks(df,'VINTAGE'), abstract_events(df_events,'VINTAGE')

In [5]:
def insert(df_matches=None,df_events=None,credentials=credentials):
    events_query = """
        INSERT INTO "EVENTS" ("EVENT_ID", "EVENT_DATE", "EVENT_TYPE_ID", "PROC_DT")
        VALUES (%s, %s, %s, %s)
        ON CONFLICT ("EVENT_ID")
        DO NOTHING
    """
    matches_query = """
        INSERT INTO "MATCHES" ("MATCH_ID", "P1", "P2", "P1_WINS", "P2_WINS", "MATCH_WINNER", "P1_DECK_ID", "P2_DECK_ID", "P1_NOTE", "P2_NOTE", "EVENT_ID", "PROC_DT")
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT ("MATCH_ID", "P1") 
        DO NOTHING
    """

    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        proc_dt = datetime.now()
        # Insert events
        if df_events is not None:
            values_list = [(row['EVENT_ID'], row['EVENT_DATE'], row['EVENT_TYPE_ID'], proc_dt) for index, row in df_events.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(events_query, values)
                except Exception as e:
                    print(f"Error inserting row into EVENTS: {values} | Error: {e}")
                    continue

        # Insert matches
        if df_matches is not None:
            values_list = [(row['MATCH_ID'], row['P1'], row['P2'], row['P1_WINS'], row['P2_WINS'],
                            row['MATCH_WINNER'], row['P1_DECK_ID'], row['P2_DECK_ID'], row['P1_NOTE'], row['P2_NOTE'], row['EVENT_ID'], proc_dt) 
                           for index, row in df_matches.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(matches_query, values)
                except Exception as e:
                    print(f"Error inserting row into MATCHES: {values} | Error: {e}")
                    continue
        conn.commit()

    except Exception as e:
        print(f"Error occurred while loading data: {e}")
        traceback.print_exc() 
        conn.rollback()

    finally:
        if conn:
            cursor.close()
            conn.close()

In [6]:
df_matches, df_events = parse_matchup_sheet(sheet_curr,gid_matches)

In [ ]:
end_date = datetime.today().date() - timedelta(days=7)
max_event_id, max_match_id = get_max_id()

insert(df_matches,df_events)

(12000000000, Timestamp('2024-08-29 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000001, Timestamp('2024-08-30 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000002, Timestamp('2024-08-31 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000003, Timestamp('2024-08-31 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000004, Timestamp('2024-09-01 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000005, Timestamp('2024-09-05 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000007, Timestamp('2024-09-07 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000008, Timestamp('2024-09-08 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 532263))
(12000000009, Timestamp('2024-09-12 00:00:00'), 14000000000, datetime.datetime(2025, 2, 10, 1, 52, 59, 5

### Test functions.

In [14]:
df_matches

,MATCH_ID,P1,P2,P1_WINS,P2_WINS,MATCH_WINNER,P1_DECK_ID,P2_DECK_ID,P1_NOTE,P2_NOTE,EVENT_ID
0,11000000000,unluckymonkey,crK,2,0,P1,13000000021,13000000028,NA,NA,12000000091
1,11000000000,crK,unluckymonkey,0,2,P2,13000000028,13000000021,NA,NA,12000000091
2,11000000044,unluckymonkey,medvedev,2,1,P1,13000000021,13000000011,NA,NA,12000000091
3,11000000023,crK,Rhianne,2,1,P1,13000000028,13000000024,NA,NA,12000000091
4,11000000023,Rhianne,crK,1,2,P2,13000000024,13000000028,NA,NA,12000000091
...,...,...,...,...,...,...,...,...,...,...,...
24055,11000012020,2plus2isfive,Zenith777,1,2,P2,13000000025,13000000007,NA,NA,12000000000
24056,11000012016,s063,unluckymonkey,0,2,P2,13000000010,13000000020,NA,NA,12000000000
24057,11000012028,OrnatePuzzles,TonyScapone,1,2,P2,13000000029,13000000027,NA,NA,12000000000
24058,11000012024,Ale_Mtg,AFX,0,2,P2,13000000015,13000000006,NA,NA,12000000000


In [ ]:
df_matches.groupby(['EVENT_ID']).agg({'P1':'nunique','EVENT_TYPE':'last','EVENT_DATE':'last'}).sort_values(by=['P1'],ascending=False).reset_index()

In [79]:
df_valid_decks.iloc[10,2] = 'POOPSDAY2'
df_valid_event_types.iloc[2,1] = 'POOPLIM2'